# 01 — Herculean auditing data cleaning

One source row is one DQC assertion occurrence for an SEC filing. This notebook reads the unchanged CSV in `data/raw`, creates only in-memory cleaned views, and displays the checks used in `cleaning_report.md`. Run cells from top to bottom. Outputs are intentionally empty in the submitted notebook.

Sources: [Herculean dataset](https://huggingface.co/datasets/TheFinAI/Herculean), [paper, Table 2](https://arxiv.org/html/2605.14355v3). The CSV was downloaded from dataset revision `52d3ecb97e490a2392717935402cde283e8fa36e` on 2026-10-05; SHA-256 is `fa52ccea5b80ea8ff1b27e5e38b5510caf74a82682858e1cc53641bbd1f7fdf2`.


In [1]:
from pathlib import Path
import hashlib
import re
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if not (project_root / "data/raw/mathematical_reasoning.csv").exists():
    project_root = Path.cwd().parents[1]
raw_path = project_root / "data/raw/mathematical_reasoning.csv"
expected_sha256 = "fa52ccea5b80ea8ff1b27e5e38b5510caf74a82682858e1cc53641bbd1f7fdf2"
actual_sha256 = hashlib.sha256(raw_path.read_bytes()).hexdigest()
assert actual_sha256 == expected_sha256, "Source file differs from the pinned download"
raw = pd.read_csv(raw_path, dtype="string", keep_default_na=False)
assert raw.shape == (362, 20)
display(raw.head(5))
print("rows, columns:", raw.shape, "SHA-256:", actual_sha256)


,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,report.base-taxonomy,report.creation-software,report.document-type,report.entry-url,report.filing-date,report.filing-year,report.id,report.sec-url,report.sic-code,folder_name
0,DQC.US.0126.9590,"2022-03-09 20:00:03,338 [DQC.US.0126.9590] The...",2022-03-10,ERROR,DQC,126,1383088,CYTTA CORP.,2022-01-12 13:46:00,0001477932-22-000209,US GAAP 2020,XBRL Document Created with XBRLMaster,10-K,http://www.sec.gov/Archives/edgar/data/1383088...,2022-01-12,2022,411725,https://www.sec.gov/Archives/edgar/data/138308...,7372,10k-cyca-20210930
1,DQC.US.0126.9590,"2022-03-09 20:00:03,339 [DQC.US.0126.9590] The...",2022-03-10,ERROR,DQC,126,1383088,CYTTA CORP.,2022-01-12 13:46:00,0001477932-22-000209,US GAAP 2020,XBRL Document Created with XBRLMaster,10-K,http://www.sec.gov/Archives/edgar/data/1383088...,2022-01-12,2022,411725,https://www.sec.gov/Archives/edgar/data/138308...,7372,10k-cyca-20210930
2,DQC.US.0126.9590,"2022-03-04 20:32:57,578 [DQC.US.0126.9590] The...",2022-03-05,ERROR,DQC,126,1766352,"Carrier EQ, LLC",2022-01-21 19:05:00,0001553350-22-000052,US GAAP 2021,Field: Set; Name: xdx; ID: xdx_06B_USD_1_iso42...,10-K,http://www.sec.gov/Archives/edgar/data/1766352...,2022-01-24,2022,413266,https://www.sec.gov/Archives/edgar/data/176635...,7372,10k-airfox-20210930
3,DQC.US.0126.9590,"2022-03-04 20:32:57,579 [DQC.US.0126.9590] The...",2022-03-05,ERROR,DQC,126,1766352,"Carrier EQ, LLC",2022-01-21 19:05:00,0001553350-22-000052,US GAAP 2021,Field: Set; Name: xdx; ID: xdx_06B_USD_1_iso42...,10-K,http://www.sec.gov/Archives/edgar/data/1766352...,2022-01-24,2022,413266,https://www.sec.gov/Archives/edgar/data/176635...,7372,10k-airfox-20210930
4,DQC.US.0126.9590,"2022-03-09 20:10:36,329 [DQC.US.0126.9590] The...",2022-03-10,ERROR,DQC,126,712515,ELECTRONIC ARTS INC.,2022-02-08 16:06:00,0000712515-22-000003,US GAAP 2021,XBRL Document Created with Wdesk from Workiva,10-Q,http://www.sec.gov/Archives/edgar/data/712515/...,2022-02-08,2022,416925,https://www.sec.gov/Archives/edgar/data/712515...,7372,10q-ea-20211231


rows, columns: (362, 20) SHA-256: fa52ccea5b80ea8ff1b27e5e38b5510caf74a82682858e1cc53641bbd1f7fdf2


## Grain, types, and conversion failures

Keep CIK, SIC, accession, report ID, and assertion type as text. Their digits encode identifiers or categories, not quantities; accession numbers contain leading zeros. Convert only dates, timestamp, and filing year for analysis. A report can contain multiple assertions, so accession alone is not a row key. The composite `(report.accession, assertion.code, assertion.detail)` is unique in this release.


In [2]:
date_columns = ["assertion.run-date", "report.filing-date", "report.accepted-timestamp"]
clean = raw.copy()
failures = {}
for column in date_columns:
    parsed = pd.to_datetime(raw[column], errors="coerce")
    failures[column] = int((parsed.isna() & raw[column].ne("")).sum())
    clean[column] = parsed
clean["report.filing-year"] = pd.to_numeric(raw["report.filing-year"], errors="coerce").astype("Int64")
failures["report.filing-year"] = int((clean["report.filing-year"].isna() & raw["report.filing-year"].ne("")).sum())
display(pd.DataFrame({"source_dtype": raw.dtypes.astype(str), "clean_dtype": clean.dtypes.astype(str)}))
display(pd.Series(failures, name="failed_nonblank_conversions").to_frame())
assert sum(failures.values()) == 0


,source_dtype,clean_dtype
assertion.code,string,string
assertion.detail,string,string
assertion.run-date,string,datetime64[us]
assertion.severity,string,string
assertion.source,string,string
assertion.type,string,string
entity.cik,string,string
entity.name,string,string
report.accepted-timestamp,string,datetime64[us]
report.accession,string,string


,failed_nonblank_conversions
assertion.run-date,0
report.filing-date,0
report.accepted-timestamp,0
report.filing-year,0


## Text and duplicate review

Do not case-fold legal entity names, assertion details, codes, or URLs. Strip surrounding whitespace from display text only; preserve the raw fields and internal whitespace in assertion details, which describes calculation evidence. In this release no values change under stripping.


In [3]:
text_columns = list(raw.columns.difference(date_columns + ["report.filing-year"]))
text_review = pd.DataFrame({"changed_by_strip": {c: int(raw[c].ne(raw[c].str.strip()).sum()) for c in text_columns}})
display(text_review)
display(pd.DataFrame({"column": ["entity.name", "assertion.code", "assertion.detail"], "before": [raw.iloc[0][c][:100] for c in ["entity.name", "assertion.code", "assertion.detail"]], "after": [raw.iloc[0][c].strip()[:100] for c in ["entity.name", "assertion.code", "assertion.detail"]]}))


,changed_by_strip
assertion.code,0
assertion.detail,0
assertion.severity,0
assertion.source,0
assertion.type,0
entity.cik,0
entity.name,0
folder_name,0
report.accession,0
report.base-taxonomy,0


,column,before,after
0,entity.name,CYTTA CORP.,CYTTA CORP.
1,assertion.code,DQC.US.0126.9590,DQC.US.0126.9590
2,assertion.detail,"2022-03-09 20:00:03,338 [DQC.US.0126.9590] The...","2022-03-09 20:00:03,338 [DQC.US.0126.9590] The..."


In [4]:
key = ["report.accession", "assertion.code", "assertion.detail"]
repeat_review = pd.DataFrame({"measure": ["exact duplicate rows", "repeated report IDs (extra rows)", "repeated accession+code (extra rows)", "duplicate composite keys", "unique filings", "unique assertions"], "count": [int(raw.duplicated().sum()), int(raw["report.id"].duplicated().sum()), int(raw.duplicated(["report.accession", "assertion.code"]).sum()), int(raw.duplicated(key).sum()), int(raw["report.accession"].nunique()), len(raw)]})
display(repeat_review)
assert not raw.duplicated().any() and not raw.duplicated(key).any()


,measure,count
0,exact duplicate rows,0
1,repeated report IDs (extra rows),204
2,repeated accession+code (extra rows),174
3,duplicate composite keys,0
4,unique filings,158
5,unique assertions,362


## Missing values and suspicious records

Treat empty strings and whitespace-only strings as missing for the audit. No such values occur. Preserve every record. A filing date after the acceptance date can reflect SEC filing-date rules and is flagged for review rather than overwritten. `folder_name` is not a unique filing key.


In [5]:
missing_before = raw.apply(lambda col: int(col.str.strip().eq("").sum()))
missing_after = clean.apply(lambda col: int(col.isna().sum()) if pd.api.types.is_datetime64_any_dtype(col) or str(col.dtype) == "Int64" else int(col.str.strip().eq("").sum()))
display(pd.DataFrame({"before": missing_before, "after": missing_after}))
assert missing_before.sum() == missing_after.sum() == 0


,before,after
assertion.code,0,0
assertion.detail,0,0
assertion.run-date,0,0
assertion.severity,0,0
assertion.source,0,0
assertion.type,0,0
entity.cik,0,0
entity.name,0,0
report.accepted-timestamp,0,0
report.accession,0,0


In [6]:
flagged = pd.DataFrame(index=raw.index)
flagged["invalid_date"] = clean[date_columns].isna().any(axis=1)
flagged["filing_year_mismatch"] = clean["report.filing-year"].ne(clean["report.filing-date"].dt.year)
flagged["run_before_filing"] = clean["assertion.run-date"].lt(clean["report.filing-date"])
flagged["accepted_date_differs"] = clean["report.accepted-timestamp"].dt.date.ne(clean["report.filing-date"].dt.date)
flagged["bad_accession_format"] = ~raw["report.accession"].str.fullmatch(r"\d{10}-\d{2}-\d{6}")
flagged["bad_cik_format"] = ~raw["entity.cik"].str.fullmatch(r"\d{1,10}")
flagged["bad_document_type"] = ~raw["report.document-type"].isin(["10-K", "10-Q"])
display(flagged.sum().rename("flagged_rows").to_frame())
display(raw.loc[flagged["accepted_date_differs"], ["report.accession", "report.accepted-timestamp", "report.filing-date"]].head(5))
folder_filings = raw.groupby("folder_name")["report.accession"].nunique()
display(folder_filings[folder_filings.gt(1)].rename("distinct_filings").to_frame())
assert not flagged[["invalid_date", "filing_year_mismatch", "run_before_filing", "bad_accession_format", "bad_cik_format", "bad_document_type"]].any().any()


,flagged_rows
invalid_date,0
filing_year_mismatch,0
run_before_filing,0
accepted_date_differs,43
bad_accession_format,0
bad_cik_format,0
bad_document_type,0


,report.accession,report.accepted-timestamp,report.filing-date
2,0001553350-22-000052,2022-01-21 19:05:00,2022-01-24
3,0001553350-22-000052,2022-01-21 19:05:00,2022-01-24
13,0001690680-22-000011,2022-02-28 17:46:00,2022-03-01
15,0001564590-22-008597,2022-03-03 19:40:00,2022-03-04
16,0001564590-22-008597,2022-03-03 19:40:00,2022-03-04


,distinct_filings
folder_name,
10k-apa-20211231,2


## Reconciliation and calculation denominators

No rows are dropped, imputed, or merged. Assertion counts use 362 rows. Filing counts deduplicate accessions and use 158 filings. Group keys have no missing values. Table 2 summarizes 65 audited instances per configuration; its percentages are separate from the 362 source assertions.


In [7]:
reconcile = pd.DataFrame({"stage": ["source assertion rows", "rows removed", "clean assertion rows", "source filings", "clean filings"], "count": [len(raw), 0, len(clean), raw["report.accession"].nunique(), clean["report.accession"].nunique()]})
display(reconcile)
display(clean.groupby("report.document-type", dropna=False).agg(assertion_rows=("assertion.code", "size"), filings=("report.accession", "nunique")))
display(clean.groupby("assertion.type", dropna=False).agg(assertion_rows=("assertion.code", "size")))
assert len(clean) == 362 and clean["report.accession"].nunique() == 158


,stage,count
0,source assertion rows,362
1,rows removed,0
2,clean assertion rows,362
3,source filings,158
4,clean filings,158


,assertion_rows,filings
report.document-type,,
10-K,145,58
10-Q,217,100


,assertion_rows
assertion.type,
117,136
126,110
15,116


## Five record checks: expected results recorded before inspection

Row numbers refer to the source CSV header plus one-based data-row positions. These checks cover ordinary 10-K and 10-Q records, an acceptance/filing date difference, a repeated accession-and-rule pair, and a numeric-looking category. Expected values were set from the source inspection before comparing against the cleaned view.


In [8]:
expected_cases = pd.DataFrame([
    {"source_row": 1, "field": "report.accession", "original": "0001477932-22-000209", "expected": "0001477932-22-000209", "reason": "leading zeros in accession must remain"},
    {"source_row": 3, "field": "report.filing-date", "original": "2022-01-24", "expected": "2022-01-24", "reason": "keep official filing date despite earlier acceptance"},
    {"source_row": 5, "field": "report.document-type", "original": "10-Q", "expected": "10-Q", "reason": "quarterly filing category is retained"},
    {"source_row": 71, "field": "assertion.code", "original": "DQC.US.0117.9574", "expected": "DQC.US.0117.9574", "reason": "repeated accession+rule still has distinct detail"},
    {"source_row": 102, "field": "assertion.type", "original": "15", "expected": "15", "reason": "numeric-looking assertion type remains a category"},
])
expected_cases


,source_row,field,original,expected,reason
0,1,report.accession,0001477932-22-000209,0001477932-22-000209,leading zeros in accession must remain
1,3,report.filing-date,2022-01-24,2022-01-24,keep official filing date despite earlier acce...
2,5,report.document-type,10-Q,10-Q,quarterly filing category is retained
3,71,assertion.code,DQC.US.0117.9574,DQC.US.0117.9574,repeated accession+rule still has distinct detail
4,102,assertion.type,15,15,numeric-looking assertion type remains a category


In [9]:
def show_value(value):
    if isinstance(value, pd.Timestamp):
        return value.strftime("%Y-%m-%d %H:%M:%S" if value.hour or value.minute or value.second else "%Y-%m-%d")
    return str(value)
checks = expected_cases.copy()
checks["actual"] = [show_value(clean.iloc[int(case.source_row)-1][case.field]) for case in checks.itertuples(index=False)]
checks["source_matches"] = [str(raw.iloc[int(case.source_row)-1][case.field]) == case.original for case in checks.itertuples(index=False)]
checks["matches_expected"] = checks["actual"].eq(checks["expected"]) & checks["source_matches"]
display(checks)
assert checks["matches_expected"].all()


,source_row,field,original,expected,reason,actual,source_matches,matches_expected
0,1,report.accession,0001477932-22-000209,0001477932-22-000209,leading zeros in accession must remain,0001477932-22-000209,True,True
1,3,report.filing-date,2022-01-24,2022-01-24,keep official filing date despite earlier acce...,2022-01-24,True,True
2,5,report.document-type,10-Q,10-Q,quarterly filing category is retained,10-Q,True,True
3,71,assertion.code,DQC.US.0117.9574,DQC.US.0117.9574,repeated accession+rule still has distinct detail,DQC.US.0117.9574,True,True
4,102,assertion.type,15,15,numeric-looking assertion type remains a category,15,True,True


## Paper Table 2: auditing configuration summary

The next cell transcribes the Auditing rows of [Table 2](https://arxiv.org/html/2605.14355v3), in its printed framework/model order. Percentages are paper-level aggregates for 65 instances per configuration. They cannot be joined to individual CSV assertions because no case-level predictions or run IDs were published.


In [10]:
frameworks = ["ReAct Agent", "Claude Code", "Codex", "Hermes", "OpenClaw"]
models = ["sonnet", "gpt", "qwen397", "qwen27"]
configurations = [(framework, model) for framework in frameworks for model in models]
paper = pd.DataFrame(configurations, columns=["framework", "model"])
paper["ACC_pct"] = [20.00,3.08,15.38,18.46,66.15,44.62,36.92,43.08,63.08,63.08,49.23,27.69,20.00,20.00,20.00,16.92,66.15,66.15,43.08,36.92]
paper["SER_pct"] = [80.00,80.00,80.00,80.00,0.00,0.00,3.08,0.00,0.00,0.00,0.00,44.62,80.00,80.00,80.00,80.00,0.00,0.00,0.00,0.00]
paper["EER_pct"] = [0.00,0.00,0.00,0.00,6.15,6.15,9.23,12.31,6.15,6.15,6.15,7.69,0.00,0.00,0.00,0.00,6.15,6.15,10.77,7.69]
paper["CER_pct"] = [0.00,16.92,4.62,1.54,27.69,49.23,50.77,44.62,30.77,30.77,44.62,20.00,0.00,0.00,0.00,3.08,27.69,27.69,46.15,55.38]
paper["audit_instances"] = 65
paper["rate_sum_pct"] = paper[["ACC_pct", "SER_pct", "EER_pct", "CER_pct"]].sum(axis=1)
display(paper)
print("largest rounding difference from 100%:", (paper["rate_sum_pct"]-100).abs().max())
assert len(paper) == 20 and not paper.duplicated(["framework", "model"]).any()
assert (paper["rate_sum_pct"]-100).abs().le(0.02).all()


,framework,model,ACC_pct,SER_pct,EER_pct,CER_pct,audit_instances,rate_sum_pct
0,ReAct Agent,sonnet,20.00,80.00,0.00,0.00,65,100.00
1,ReAct Agent,gpt,3.08,80.00,0.00,16.92,65,100.00
2,ReAct Agent,qwen397,15.38,80.00,0.00,4.62,65,100.00
3,ReAct Agent,qwen27,18.46,80.00,0.00,1.54,65,100.00
4,Claude Code,sonnet,66.15,0.00,6.15,27.69,65,99.99
5,Claude Code,gpt,44.62,0.00,6.15,49.23,65,100.00
6,Claude Code,qwen397,36.92,3.08,9.23,50.77,65,100.00
7,Claude Code,qwen27,43.08,0.00,12.31,44.62,65,100.01
8,Codex,sonnet,63.08,0.00,6.15,30.77,65,100.00
9,Codex,gpt,63.08,0.00,6.15,30.77,65,100.00


largest rounding difference from 100%: 0.009999999999990905


## Reproducibility and limits

The CSV lacks case-level predictions, cost, latency, and tool traces. The 362 assertions are not the 65 benchmark instances per configuration. Table 2 rates must stay at configuration grain, and 43 filing/acceptance date differences remain flags pending SEC-level verification.
